In [1]:
import os
import sys
import arrow
import numpy as np
import pandas as pd
from math import pi
import matplotlib.pyplot as plt
import matplotlib.dates as dates
import seaborn as sns
from datetime import datetime
from numpy import nan_to_num
from matplotlib.colors import LogNorm
from matplotlib.ticker import ScalarFormatter
import os
import sys
import arrow
import numpy as np
import pandas as pd
from math import pi
import matplotlib.pyplot as plt
import matplotlib.dates as dates
import seaborn as sns
from datetime import date
from numpy import nan_to_num
from matplotlib.colors import LogNorm
from matplotlib.ticker import ScalarFormatter
import pandas as pd
from datetime import datetime, timedelta

STEP 0: Create the dfs for size_dist_diameter_input, datetime, and size_dist_input csvs needed for binning

In [2]:
#Generate size_dist_number.csv and size_dist_diameter.csv for MODULE A
AEROSOL_DIAMS = [150, 169.8, 192.1, 217.5, 246.1, 278.6, 315.3, 356.8, 403.9, 457.1, 517.3, 585.5, 662.7, 750]

#read newly date time fixed NASA file
#df_may = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Model for may dataset\ModuleA\test.csv")
df_may = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\MAC_DATASET_LLOD_FILTERED_V3_10campaigns.csv")

#HERE: DATETIME
df_datetime = df_may[['datetime']].copy()
df_datetime = df_datetime.rename(columns={'datetime': 'datetime_all'})

#POPULATE size_dist_diameter_input.csv
df = pd.DataFrame({'diameter': AEROSOL_DIAMS})

#HERE: SIZE_DIST_DIAMETER_INPUT 
df_size_dist_diameter_input = df.copy()

#POPULATE size_dist_input.csv
# Select only columns that start with "bin"
bin_columns = [col for col in df_may.columns if col.startswith('bin')]

# Create a new DataFrame with only the bin columns
df_bins = df_may[bin_columns]

# Drop the last bin column
df_bins = df_bins.iloc[:, :-1]

# Transpose the DataFrame (rows = diameters, columns = time steps)
df_bins_transposed = df_bins.transpose()

#HERE: SIZE_DIST_NUMBER
# Reset the index so that "bin1", "bin2", etc., are not saved as extra rows/columns
df_bins_transposed = df_bins_transposed.reset_index(drop=True)
df_size_dist_input = df_bins_transposed.copy()


STEP 1: Run binning algorithm borrowed from Module A

In [3]:
df_MAC = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\MAC_DATASET_LLOD_FILTERED_V3_10campaigns.csv")

#HERE: Change the diameters for Number and Volume
d_Nx_list = [250.0, 500.0]
d_Vx_list = [250.0, 500.0]

#How many bins per N and V
len_n = len(d_Nx_list)
len_v = len(d_Vx_list)

#function purpose: find nearest value (d_N1, d_N2, d_V1, d_V2) in numpy array (instrument measured diameters)
def find_nearest(array, value):
    array = [float(i) for i in array]  # force float
    n = [abs(i-value) for i in array]
    idx = n.index(min(n))
    return idx

#Read size_dist_input.csv and size_dist_diameter_input.csv
dNdlogdP = df_size_dist_input.values
diameter = df_size_dist_diameter_input['diameter']

# Mask columns that are NOT all NaN
mask = ~np.all(np.isnan(dNdlogdP), axis=0)

# Filter out all-NaN columns
dNdlogdP = dNdlogdP[:, mask]

#convert to dataframe back
size_dist_input = pd.DataFrame(dNdlogdP)
size_dist_diameter_input = pd.DataFrame(diameter)

# make sure properly formatted
# Adopted From Cora's code
size_dist_diameter_input.reset_index(drop=True)
size_dist_diameter_input = size_dist_diameter_input.astype(float)

#find nearest index in size_dist_diameter_input based on user defined d_N & d_V
d_Nx_nearest = []
d_Vx_nearest = []

# len(d_Nx_list)=2 so in in range(0, 2)
for i in range(len(d_Nx_list)):
    x = find_nearest(size_dist_diameter_input['diameter'], d_Nx_list[i])
    d_Nx_nearest.append(x)
for i in range(len(d_Vx_list)):
    x = find_nearest(size_dist_diameter_input['diameter'], d_Vx_list[i])
    d_Vx_nearest.append(x)

# From Cora's code
# Add beginning and ending indexes for the divisions
# (bin1= 0~d_N1, bin2=d_N1~d_N2, bin3= >d_N2)
# Here, beginning index=0, ending index=30 so it will look like [0,6,12,30]
d_Nx_nearest.insert(0,0)
d_Nx_nearest.append(len(size_dist_diameter_input))  
d_Vx_nearest.insert(0,0)
d_Vx_nearest.append(len(size_dist_diameter_input))

# Need to know more why we take dN/dlog(dp)?
# Logarithmic scaling is often used in scientific data when there's a large range of values, especially when the data spans many orders of magnitude.
# It helps in visualizing data that follows a power-law distribution.
ln_size_dist_diameter_input=np.log(size_dist_diameter_input)

#replace all input values less than 0 with np.nan
#FIXME: consider LLOD of instrument instead instead of 0
size_dist_input.mask(size_dist_input <= 0, np.nan , inplace=True )

#calculate volume distribution based on diameter^3
diameter_power = np.power(size_dist_diameter_input, 3)

# making it as numpy array
diameter_power = diameter_power.to_numpy()

# Calculate size_dist_volume
# When you specify axis='index', it means that each row in the DataFrame will be multiplied by
# the corresponding value in the diameter_power Series

size_dist_volume = pi/6*size_dist_input.mul(diameter_power,axis='index')

# Make NaN value to 0
size_dist_input[np.isnan(size_dist_input)] = 0 
size_dist_volume[np.isnan(size_dist_volume)] = 0

# create empty dataframes with  (num_columns in size_dist_input) x (number_bins) 
# e.g., 5760x3: That means we are converting 30 bins into 3 bins each contains 5760 values
num_bins = len(d_Nx_list) + 1
N_df = pd.DataFrame(index = range(np.size(size_dist_input,1)), columns = range(num_bins))
F_N_df = pd.DataFrame(index = range(np.size(size_dist_input,1)), columns = range(num_bins))
V_df = pd.DataFrame(index = range(np.size(size_dist_input,1)), columns = range(num_bins))
F_V_df = pd.DataFrame(index = range(np.size(size_dist_input,1)), columns = range(num_bins))

#for each bin, make a smaller dataframe with the diameter bins desired. Then apply trapz calculation by row
#and add result to dataframe in adjacent bin
# here, num_bins=3 so, i == 0, 1, 2
for i in range(num_bins):
    if i == 0:
        #size_dist_input (30*186) > 30 bins > column contains number concentrations (dN/dlogDp)
        # for first bin(i==0), we are cutting rows=0:6, column=all in a small df
        # N_df (186*3)
        # how np.trapz works?
        # small_df has values in 6 rows > y(x); x also contains 6 diameter, so operation in one column at a time
        # When axis=0 is used: The integration will be performed along each column.
        # suppose, 6 diameter (1,2,3,4,5,6) and first column in samll_df is (10,20,30,40,50,60)
        # np.trapz> 1/2[(10+20)*(2-1)+(20+30)*(3-2)+(30+40)*(4-3)+(40+50)*(5-4)+(50+60)*(6-5)]=175
        # Trapezoidal area under the curve=(1/2)(f(x0)+f(x1))(x1-x0)
        # N_df[0,0] = 175, N_df[1,0] come from the second column
        # In this example, V_df[:,2]=0 because in the last bean there are only one row exit which contains 0 in each column.
        small_df = size_dist_input.iloc[0:(d_Nx_nearest[i+1] + 1), :]
        N_df.iloc[:, i] = np.trapz(
            small_df,
            x=ln_size_dist_diameter_input[0:(d_Nx_nearest[i+1] + 1)],
            axis=0
        )
    elif i > 0:
        small_df = size_dist_input.iloc[(d_Nx_nearest[i] + 1):(d_Nx_nearest[i+1] + 1), :]
        N_df.iloc[:, i] = np.trapz(
            small_df,
            x=ln_size_dist_diameter_input[(d_Nx_nearest[i] + 1):(d_Nx_nearest[i+1] + 1)],
            axis=0
        )

# Similarly, calculate V_df
# In this example, V_df[:,2]=0 because in the last bean there are only one row exit which contains 0 in each column.
for i in range(num_bins):
    if i == 0:
        # Include d_Vx_nearest[i+1]
        small_df = size_dist_volume.iloc[0:(d_Vx_nearest[i+1] + 1), :]
        V_df.iloc[:, i] = np.trapz(
            small_df,
            x=ln_size_dist_diameter_input[0:(d_Vx_nearest[i+1] + 1)],
            axis=0
        )
    else:
        # Include d_Vx_nearest[i+1]
        small_df = size_dist_volume.iloc[(d_Vx_nearest[i] + 1):(d_Vx_nearest[i+1] + 1), :]
        V_df.iloc[:, i] = np.trapz(
            small_df,
            x=ln_size_dist_diameter_input[(d_Vx_nearest[i] + 1):(d_Vx_nearest[i+1] + 1)],
            axis=0
        )

#calculate areaXY_number and volumeXY_number
areaXY_number = pd.Series(np.trapz(size_dist_input, x=ln_size_dist_diameter_input, axis=0))
volumeXY_number = pd.Series(np.trapz(size_dist_volume, x=ln_size_dist_diameter_input, axis=0))

#FIXME: question: can we replace 0 with np.nan?
areaXY_number.replace(0, np.nan, inplace=True)
volumeXY_number.replace(0, np.nan, inplace=True)

#divide N by areaXY for f_N dataframe
for i in range(num_bins):
    F_N_df.iloc[:,i] = N_df.iloc[:,i] / areaXY_number
for i in range(num_bins):
    F_V_df.iloc[:,i] = V_df.iloc[:,i] / volumeXY_number

#load SMPS_APS datetime
datetimedf = df_datetime.copy()
datetimedf['datetime_all'] = pd.to_datetime(datetimedf['datetime_all'], format='%Y-%m-%d %H:%M:%S')

datetimedf = datetimedf['datetime_all']
date = str(datetimedf[0])
#Extract first date from the datetime object > '2022_06_18'
date = datetime.strptime(date, '%Y-%m-%d %H:%M:%S').strftime('%Y_%m_%d')

# Put everything in big dataframe with datetime!
file_M_1 = pd.DataFrame() 
file_M_1['datetime'] = datetimedf
for i in range(num_bins):
    file_M_1['N'+str(i+1)] = N_df.iloc[:,i]
    file_M_1['F_N'+str(i+1)] = F_N_df.iloc[:,i]
    file_M_1['V'+str(i+1)] = V_df.iloc[:,i]
    file_M_1['F_V'+str(i+1)] = F_V_df.iloc[:,i]
file_M_1['V_total'] = volumeXY_number
file_M_1['N_total'] = areaXY_number


#Populate F_N_Total and F_V_Total
# Dynamically generate column names
F_N_cols = [f'F_N{i+1}' for i in range(num_bins)]
F_V_cols = [f'F_V{i+1}' for i in range(num_bins)]

# Calculate the total for F_N and F_V
file_M_1['F_N_Total'] = file_M_1[F_N_cols].sum(axis=1)
file_M_1['F_V_Total'] = file_M_1[F_V_cols].sum(axis=1)

# Create dynamic column lists
N_cols     = [f'N{i+1}' for i in range(num_bins)]
F_N_cols   = [f'F_N{i+1}' for i in range(num_bins)]
V_cols     = [f'V{i+1}' for i in range(num_bins)]
F_V_cols   = [f'F_V{i+1}' for i in range(num_bins)]

# Construct full column order
cols_order = (
    ['datetime'] +
    N_cols + ['N_total'] +
    F_N_cols +
    V_cols + ['V_total'] +
    F_V_cols +
    ['F_N_Total', 'F_V_Total']
)

# Apply to DataFrame
file_M_1 = file_M_1[cols_order]

# Step 1: Remove columns from df_MAC that exist in file_M_1
columns_to_remove = [col for col in df_MAC.columns if col in file_M_1.columns]
df_MAC_filtered = df_MAC.drop(columns=columns_to_remove)

# Step 2: Concatenate the filtered df_MAC with file_M_1
# This adds file_M_1's columns and rows to df_MAC
result_df = pd.concat([df_MAC_filtered, file_M_1], axis=1, ignore_index=False)

result_df.to_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\MAC_binning_{len_n + 1}bins.csv", index=False)

KeyboardInterrupt: 

ADDITIONAL: Optimize Bin diams by number of bins

In [ ]:
# USER PARAMETER: Set the number of bins you want
NUM_BINS_DESIRED = 6  # Change this to your desired number of bins

In [9]:
# ============================================================================
# OPTIMIZATION FUNCTION: Quantile-based bin boundary optimization
# ============================================================================
df_MAC = pd.read_csv(rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\MAC_DATASET_LLOD_FILTERED_V3_10campaigns.csv")


def optimize_bin_boundaries_multi_objective(num_bins, size_dist_input, size_dist_volume, diameter):
    """
    Multi-objective optimization:
    1. Minimize empty bins (primary objective)
    2. Achieve equal counts/volumes distribution (secondary objective)
    """
    print(f"Multi-objective optimization for {num_bins} bins...")
    print(f"Available diameter range: {diameter.min():.1f} - {diameter.max():.1f} nm")
    
    # Get data for optimization
    total_counts = np.sum(size_dist_input.values, axis=1)
    total_volumes = np.sum(size_dist_volume.values, axis=1)
    
    # Only use diameter bins with significant data
    threshold_counts = np.max(total_counts) * 0.01
    threshold_volumes = np.max(total_volumes) * 0.01
    
    valid_count_indices = total_counts > threshold_counts
    valid_volume_indices = total_volumes > threshold_volumes
    
    def evaluate_boundaries(d_list, data_totals, is_volume=False):
        """Evaluate how good a set of boundaries is"""
        if len(d_list) != num_bins - 1:
            return float('inf')  # Invalid
            
        # Find nearest indices
        nearest_indices = []
        for d_val in d_list:
            idx = find_nearest(diameter.values, d_val)
            nearest_indices.append(idx)
        
        # Add start and end
        nearest_indices.insert(0, 0)
        nearest_indices.append(len(diameter))
        
        # Calculate bin contents
        bin_contents = []
        for i in range(num_bins):
            start_idx = nearest_indices[i]
            end_idx = nearest_indices[i + 1]
            
            if start_idx == end_idx:
                content = 0
            else:
                if i == 0:
                    content = np.sum(data_totals[start_idx:end_idx])
                else:
                    content = np.sum(data_totals[start_idx+1:end_idx])
            
            bin_contents.append(content)
        
        # Objective 1: Minimize empty bins (primary - heavily weighted)
        empty_bins = sum(1 for content in bin_contents if content <= 0)
        empty_penalty = empty_bins * 1e6  # Heavy penalty for empty bins
        
        # Objective 2: Minimize variance (equal distribution)
        if all(content > 0 for content in bin_contents):
            variance = np.var(bin_contents)
            normalized_variance = variance / (np.mean(bin_contents)**2) if np.mean(bin_contents) > 0 else 1e6
        else:
            normalized_variance = 1e6  # High penalty if any bin is empty
        
        # Combined score (lower is better)
        score = empty_penalty + normalized_variance
        return score, empty_bins, bin_contents
    
    def generate_candidate_boundaries(data_totals, valid_indices, strategy='quantile'):
        """Generate candidate boundaries using different strategies"""
        valid_diameters = diameter[valid_indices]
        valid_data = data_totals[valid_indices]
        
        if strategy == 'quantile':
            # Quantile-based approach
            cumulative_data = np.cumsum(valid_data)
            cumulative_normalized = cumulative_data / cumulative_data[-1]
            
            boundaries = []
            for i in range(1, num_bins):
                target_quantile = i / num_bins
                idx = np.argmin(np.abs(cumulative_normalized - target_quantile))
                boundaries.append(float(valid_diameters.iloc[idx]))
            
            return sorted(list(set(boundaries)))  # Remove duplicates and sort
        
        elif strategy == 'equal_spacing':
            # Equal diameter spacing
            min_d = valid_diameters.min()
            max_d = valid_diameters.max()
            spacing = (max_d - min_d) / num_bins
            
            return [min_d + (i+1) * spacing for i in range(num_bins-1)]
        
        elif strategy == 'adaptive':
            # Adaptive approach: try to ensure each bin has at least some data
            boundaries = []
            data_per_bin = len(valid_diameters) // num_bins
            
            for i in range(1, num_bins):
                idx = min(i * data_per_bin, len(valid_diameters) - 1)
                boundaries.append(float(valid_diameters.iloc[idx]))
            
            return boundaries
    
    # Try multiple strategies for both counts and volumes
    strategies = ['quantile', 'equal_spacing', 'adaptive']
    
    best_count_boundaries = None
    best_count_score = float('inf')
    best_volume_boundaries = None
    best_volume_score = float('inf')
    
    print("\nTesting different boundary strategies...")
    
    # Optimize for counts
    for strategy in strategies:
        try:
            candidates = generate_candidate_boundaries(total_counts, valid_count_indices, strategy)
            if len(candidates) == num_bins - 1:  # Valid number of boundaries
                score, empty_bins, contents = evaluate_boundaries(candidates, total_counts, False)
                print(f"Count strategy '{strategy}': Score={score:.2e}, Empty bins={empty_bins}")
                
                if score < best_count_score:
                    best_count_score = score
                    best_count_boundaries = candidates
        except Exception as e:
            print(f"Count strategy '{strategy}' failed: {e}")
    
    # Optimize for volumes
    for strategy in strategies:
        try:
            candidates = generate_candidate_boundaries(total_volumes, valid_volume_indices, strategy)
            if len(candidates) == num_bins - 1:  # Valid number of boundaries
                score, empty_bins, contents = evaluate_boundaries(candidates, total_volumes, True)
                print(f"Volume strategy '{strategy}': Score={score:.2e}, Empty bins={empty_bins}")
                
                if score < best_volume_score:
                    best_volume_score = score
                    best_volume_boundaries = candidates
        except Exception as e:
            print(f"Volume strategy '{strategy}' failed: {e}")
    
    # Fallback if no good solution found
    if best_count_boundaries is None:
        print("Warning: No valid count boundaries found. Using equal spacing.")
        min_d, max_d = diameter.min(), diameter.max()
        best_count_boundaries = [min_d + (i+1) * (max_d - min_d) / num_bins for i in range(num_bins-1)]
    
    if best_volume_boundaries is None:
        print("Warning: No valid volume boundaries found. Using equal spacing.")
        min_d, max_d = diameter.min(), diameter.max()
        best_volume_boundaries = [min_d + (i+1) * (max_d - min_d) / num_bins for i in range(num_bins-1)]
    
    print(f"\nBest count boundaries: {best_count_boundaries}")
    print(f"Best volume boundaries: {best_volume_boundaries}")
    
    # Final validation
    count_score, count_empty, count_contents = evaluate_boundaries(best_count_boundaries, total_counts, False)
    volume_score, volume_empty, volume_contents = evaluate_boundaries(best_volume_boundaries, total_volumes, True)
    
    print(f"\nFinal validation:")
    print(f"Count bins - Empty: {count_empty}, Contents: {[f'{c:.2e}' for c in count_contents]}")
    print(f"Volume bins - Empty: {volume_empty}, Contents: {[f'{c:.2e}' for c in volume_contents]}")
    
    return best_count_boundaries, best_volume_boundaries

# ============================================================================
# STEP 1: Run binning algorithm with optimization
# ============================================================================

#function purpose: find nearest value (d_N1, d_N2, d_V1, d_V2) in numpy array (instrument measured diameters)
def find_nearest(array, value):
    array = [float(i) for i in array]  # force float
    n = [abs(i-value) for i in array]
    idx = n.index(min(n))
    return idx

#Read size_dist_input.csv and size_dist_diameter_input.csv
dNdlogdP = df_size_dist_input.values
diameter = df_size_dist_diameter_input['diameter']

# Mask columns that are NOT all NaN
mask = ~np.all(np.isnan(dNdlogdP), axis=0)

# Filter out all-NaN columns
dNdlogdP = dNdlogdP[:, mask]

#convert to dataframe back
size_dist_input = pd.DataFrame(dNdlogdP)
size_dist_diameter_input = pd.DataFrame(diameter)

# make sure properly formatted
# Adopted From Cora's code
size_dist_diameter_input.reset_index(drop=True)
size_dist_diameter_input = size_dist_diameter_input.astype(float)

#replace all input values less than 0 with np.nan
#FIXME: consider LLOD of instrument instead instead of 0
size_dist_input.mask(size_dist_input <= 0, np.nan , inplace=True )

#calculate volume distribution based on diameter^3
diameter_power = np.power(size_dist_diameter_input, 3)

# making it as numpy array
diameter_power = diameter_power.to_numpy()

# Calculate size_dist_volume
# When you specify axis='index', it means that each row in the DataFrame will be multiplied by
# the corresponding value in the diameter_power Series

size_dist_volume = pi/6*size_dist_input.mul(diameter_power,axis='index')

# Make NaN value to 0 for optimization calculations
size_dist_input_for_opt = size_dist_input.copy()
size_dist_volume_for_opt = size_dist_volume.copy()
size_dist_input_for_opt[np.isnan(size_dist_input_for_opt)] = 0 
size_dist_volume_for_opt[np.isnan(size_dist_volume_for_opt)] = 0

# ============================================================================
# OPTIMIZATION: Find optimal bin boundaries
# ============================================================================

d_Nx_list, d_Vx_list = optimize_bin_boundaries_multi_objective(
    NUM_BINS_DESIRED, 
    size_dist_input_for_opt, 
    size_dist_volume_for_opt, 
    size_dist_diameter_input['diameter']
)

#How many bins per N and V
len_n = len(d_Nx_list)
len_v = len(d_Vx_list)
num_bins = len(d_Nx_list) + 1

print(f"\nFinal configuration:")
print(f"Number of bins: {num_bins}")
print(f"len_n: {len_n}, len_v: {len_v}")

# Continue with original algorithm using optimized boundaries
# Make NaN value to 0 for integration
size_dist_input[np.isnan(size_dist_input)] = 0 
size_dist_volume[np.isnan(size_dist_volume)] = 0

# Need to know more why we take dN/dlog(dp)?
# Logarithmic scaling is often used in scientific data when there's a large range of values, especially when the data spans many orders of magnitude.
# It helps in visualizing data that follows a power-law distribution.
ln_size_dist_diameter_input=np.log(size_dist_diameter_input)

#find nearest index in size_dist_diameter_input based on user defined d_N & d_V
d_Nx_nearest = []
d_Vx_nearest = []

# len(d_Nx_list)=2 so in in range(0, 2)
for i in range(len(d_Nx_list)):
    x = find_nearest(size_dist_diameter_input['diameter'], d_Nx_list[i])
    d_Nx_nearest.append(x)
for i in range(len(d_Vx_list)):
    x = find_nearest(size_dist_diameter_input['diameter'], d_Vx_list[i])
    d_Vx_nearest.append(x)

# From Cora's code
# Add beginning and ending indexes for the divisions
# (bin1= 0~d_N1, bin2=d_N1~d_N2, bin3= >d_N2)
# Here, beginning index=0, ending index=30 so it will look like [0,6,12,30]
d_Nx_nearest.insert(0,0)
d_Nx_nearest.append(len(size_dist_diameter_input))  
d_Vx_nearest.insert(0,0)
d_Vx_nearest.append(len(size_dist_diameter_input))

print(f"d_Nx_nearest indices: {d_Nx_nearest}")
print(f"d_Vx_nearest indices: {d_Vx_nearest}")

# create empty dataframes with  (num_columns in size_dist_input) x (number_bins) 
# e.g., 5760x3: That means we are converting 30 bins into 3 bins each contains 5760 values
N_df = pd.DataFrame(index = range(np.size(size_dist_input,1)), columns = range(num_bins))
F_N_df = pd.DataFrame(index = range(np.size(size_dist_input,1)), columns = range(num_bins))
V_df = pd.DataFrame(index = range(np.size(size_dist_input,1)), columns = range(num_bins))
F_V_df = pd.DataFrame(index = range(np.size(size_dist_input,1)), columns = range(num_bins))

#for each bin, make a smaller dataframe with the diameter bins desired. Then apply trapz calculation by row
#and add result to dataframe in adjacent bin
# here, num_bins=3 so, i == 0, 1, 2
for i in range(num_bins):
    if i == 0:
        #size_dist_input (30*186) > 30 bins > column contains number concentrations (dN/dlogDp)
        # for first bin(i==0), we are cutting rows=0:6, column=all in a small df
        # N_df (186*3)
        # how np.trapz works?
        # small_df has values in 6 rows > y(x); x also contains 6 diameter, so operation in one column at a time
        # When axis=0 is used: The integration will be performed along each column.
        # suppose, 6 diameter (1,2,3,4,5,6) and first column in samll_df is (10,20,30,40,50,60)
        # np.trapz> 1/2[(10+20)*(2-1)+(20+30)*(3-2)+(30+40)*(4-3)+(40+50)*(5-4)+(50+60)*(6-5)]=175
        # Trapezoidal area under the curve=(1/2)(f(x0)+f(x1))(x1-x0)
        # N_df[0,0] = 175, N_df[1,0] come from the second column
        # In this example, V_df[:,2]=0 because in the last bean there are only one row exit which contains 0 in each column.
        small_df = size_dist_input.iloc[0:(d_Nx_nearest[i+1] + 1), :]
        N_df.iloc[:, i] = np.trapz(
            small_df,
            x=ln_size_dist_diameter_input[0:(d_Nx_nearest[i+1] + 1)],
            axis=0
        )
    elif i > 0:
        small_df = size_dist_input.iloc[(d_Nx_nearest[i] + 1):(d_Nx_nearest[i+1] + 1), :]
        N_df.iloc[:, i] = np.trapz(
            small_df,
            x=ln_size_dist_diameter_input[(d_Nx_nearest[i] + 1):(d_Nx_nearest[i+1] + 1)],
            axis=0
        )

# Similarly, calculate V_df
# In this example, V_df[:,2]=0 because in the last bean there are only one row exit which contains 0 in each column.
for i in range(num_bins):
    if i == 0:
        # Include d_Vx_nearest[i+1]
        small_df = size_dist_volume.iloc[0:(d_Vx_nearest[i+1] + 1), :]
        V_df.iloc[:, i] = np.trapz(
            small_df,
            x=ln_size_dist_diameter_input[0:(d_Vx_nearest[i+1] + 1)],
            axis=0
        )
    else:
        # Include d_Vx_nearest[i+1]
        small_df = size_dist_volume.iloc[(d_Vx_nearest[i] + 1):(d_Vx_nearest[i+1] + 1), :]
        V_df.iloc[:, i] = np.trapz(
            small_df,
            x=ln_size_dist_diameter_input[(d_Vx_nearest[i] + 1):(d_Vx_nearest[i+1] + 1)],
            axis=0
        )

#calculate areaXY_number and volumeXY_number
areaXY_number = pd.Series(np.trapz(size_dist_input, x=ln_size_dist_diameter_input, axis=0))
volumeXY_number = pd.Series(np.trapz(size_dist_volume, x=ln_size_dist_diameter_input, axis=0))

#FIXME: question: can we replace 0 with np.nan?
areaXY_number.replace(0, np.nan, inplace=True)
volumeXY_number.replace(0, np.nan, inplace=True)

#divide N by areaXY for f_N dataframe
for i in range(num_bins):
    F_N_df.iloc[:,i] = N_df.iloc[:,i] / areaXY_number
for i in range(num_bins):
    F_V_df.iloc[:,i] = V_df.iloc[:,i] / volumeXY_number

#load SMPS_APS datetime
datetimedf = df_datetime.copy()
datetimedf['datetime_all'] = pd.to_datetime(datetimedf['datetime_all'], format='%Y-%m-%d %H:%M:%S')

datetimedf = datetimedf['datetime_all']
date = str(datetimedf[0])
#Extract first date from the datetime object > '2022_06_18'
date = datetime.strptime(date, '%Y-%m-%d %H:%M:%S').strftime('%Y_%m_%d')

# Put everything in big dataframe with datetime!
file_M_1 = pd.DataFrame() 
file_M_1['datetime'] = datetimedf
for i in range(num_bins):
    file_M_1['N'+str(i+1)] = N_df.iloc[:,i]
    file_M_1['F_N'+str(i+1)] = F_N_df.iloc[:,i]
    file_M_1['V'+str(i+1)] = V_df.iloc[:,i]
    file_M_1['F_V'+str(i+1)] = F_V_df.iloc[:,i]
file_M_1['V_total'] = volumeXY_number
file_M_1['N_total'] = areaXY_number

#Populate F_N_Total and F_V_Total
# Dynamically generate column names
F_N_cols = [f'F_N{i+1}' for i in range(num_bins)]
F_V_cols = [f'F_V{i+1}' for i in range(num_bins)]

# Calculate the total for F_N and F_V
file_M_1['F_N_Total'] = file_M_1[F_N_cols].sum(axis=1)
file_M_1['F_V_Total'] = file_M_1[F_V_cols].sum(axis=1)

# Create dynamic column lists
N_cols     = [f'N{i+1}' for i in range(num_bins)]
F_N_cols   = [f'F_N{i+1}' for i in range(num_bins)]
V_cols     = [f'V{i+1}' for i in range(num_bins)]
F_V_cols   = [f'F_V{i+1}' for i in range(num_bins)]

# Construct full column order
cols_order = (
    ['datetime'] +
    N_cols + ['N_total'] +
    F_N_cols +
    V_cols + ['V_total'] +
    F_V_cols +
    ['F_N_Total', 'F_V_Total']
)

# Apply to DataFrame
file_M_1 = file_M_1[cols_order]

# Step 1: Remove columns from df_MAC that exist in file_M_1
columns_to_remove = [col for col in df_MAC.columns if col in file_M_1.columns]
df_MAC_filtered = df_MAC.drop(columns=columns_to_remove)

# Step 2: Concatenate the filtered df_MAC with file_M_1
# This adds file_M_1's columns and rows to df_MAC
result_df = pd.concat([df_MAC_filtered, file_M_1], axis=1, ignore_index=False)

# ============================================================================
# RESULTS AND OUTPUT
# ============================================================================

print(f"\n" + "="*60)
print("OPTIMIZATION RESULTS")
print("="*60)
print(f"Desired number of bins: {NUM_BINS_DESIRED}")
print(f"Actual number of bins created: {num_bins}")
print(f"\nOptimized d_Nx_list (equal particle counts): {d_Nx_list}")
print(f"Optimized d_Vx_list (equal volumes): {d_Vx_list}")

# Check for empty bins and calculate percentages
print(f"\nBin validation:")
total_n = sum([N_df.iloc[:, i].sum() for i in range(num_bins)])
total_v = sum([V_df.iloc[:, i].sum() for i in range(num_bins)])

for i in range(num_bins):
    n_sum = N_df.iloc[:, i].sum()
    v_sum = V_df.iloc[:, i].sum()
    n_percent = (n_sum / total_n * 100) if total_n > 0 else 0
    v_percent = (v_sum / total_v * 100) if total_v > 0 else 0
    print(f"Bin {i+1}: N_sum = {n_sum:.2e} ({n_percent:.1f}%), V_sum = {v_sum:.2e} ({v_percent:.1f}%)")

print(f"\nColumns in final DataFrame:")
print(f"Total columns: {len(result_df.columns)}")
print(f"Binning columns: {[col for col in result_df.columns if any(col.startswith(prefix) for prefix in ['N', 'F_N', 'V', 'F_V'])]}")

# Save results
output_filename = rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\binning_datasets\MAC_binning_{NUM_BINS_DESIRED}bins_optimized.csv"
result_df.to_csv(output_filename, index=False)
print(f"\nResults saved to: {output_filename}")

# Optional: Save optimization parameters for reference
opt_params_filename = rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\binning_datasets\optimization_parameters_{NUM_BINS_DESIRED}bins.txt"
with open(opt_params_filename, 'w') as f:
    f.write(f"Optimization Parameters\n")
    f.write(f"=====================\n")
    f.write(f"Desired bins: {NUM_BINS_DESIRED}\n")
    f.write(f"Actual bins: {num_bins}\n")
    f.write(f"d_Nx_list: {d_Nx_list}\n")
    f.write(f"d_Vx_list: {d_Vx_list}\n")
    f.write(f"Date processed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

print(f"Optimization parameters saved to: {opt_params_filename}")
print("="*60)

Multi-objective optimization for 5 bins...
Available diameter range: 150.0 - 750.0 nm

Testing different boundary strategies...
Count strategy 'quantile': Score=4.00e+06, Empty bins=3
Count strategy 'equal_spacing': Score=2.00e+06, Empty bins=1
Count strategy 'adaptive': Score=1.36e+00, Empty bins=0
Volume strategy 'quantile': Score=2.00e+06, Empty bins=1
Volume strategy 'equal_spacing': Score=7.73e-01, Empty bins=0
Volume strategy 'adaptive': Score=1.32e-01, Empty bins=0

Best count boundaries: [192.1, 246.1, 315.3, 403.9]
Best volume boundaries: [192.1, 246.1, 315.3, 403.9]

Final validation:
Count bins - Empty: 0, Contents: ['1.07e+10', '2.09e+09', '1.74e+09', '1.09e+09', '5.72e+08']
Volume bins - Empty: 0, Contents: ['2.06e+16', '1.13e+16', '1.97e+16', '2.58e+16', '3.64e+16']

Final configuration:
Number of bins: 5
len_n: 4, len_v: 4
d_Nx_nearest indices: [0, 2, 4, 6, 8, 14]
d_Vx_nearest indices: [0, 2, 4, 6, 8, 14]

OPTIMIZATION RESULTS
Desired number of bins: 5
Actual number of b